# Chapter 5: Python for ML, Part 3 – Advanced Python
**Companion notebook** · Book 1: Python and Data Foundations for ML

Every code example from Chapter 5, in the same order as the book. The topics are ordered so each one uses only ideas already covered.

Examples that are *meant* to cause an error are included as comments (lines starting with `#`).

## 1. More on data structures
### Set operations

In [ ]:
called_in_may = {"Priya", "Arjun", "Meera", "Ravi"}
subscribed = {"Meera", "Ravi", "Sana"}

print(sorted(called_in_may | subscribed))
print(sorted(called_in_may & subscribed))
print(sorted(called_in_may - subscribed))
print(sorted(subscribed - called_in_may))
print(sorted(called_in_may ^ subscribed))

### Nested data

In [ ]:
branch = {
    "name": "Lisbon Central",
    "customers": [
        {"name": "Priya", "balance": 1506, "loans": ["housing"]},
        {"name": "Arjun", "balance": 2143, "loans": []},
        {"name": "Meera", "balance": 102127, "loans": ["housing", "personal"]},
    ],
}

print(branch["customers"][2]["name"])
print(branch["customers"][2]["loans"][1])

for customer in branch["customers"]:
    print(customer["name"], "has", len(customer["loans"]), "loan(s)")

### Star unpacking

In [ ]:
balances = [2143, 29, 2, 1506, 1]
first, *middle, last = balances
print(first, middle, last)

def describe(name, age, job):
    return f"{name}, {age}, {job}"

details = ["Priya", 33, "technician"]
print(describe(*details))

info = {"name": "Arjun", "age": 58, "job": "management"}
print(describe(**info))

combined = [*balances, 231]
print(combined)
merged = {**info, "balance": 2143}
print(merged)

### match and case

In [ ]:
def contact_message(contact):
    match contact:
        case "cellular":
            return "Call the mobile number"
        case "telephone":
            return "Call the landline"
        case "unknown":
            return "Find a contact number first"
        case _:
            return "Unexpected contact type"

for c in ["cellular", "telephone", "unknown", "email"]:
    print(c, "->", contact_message(c))

## 2. The collections module and itertools
### Counter (needs an internet connection to download the data)

In [ ]:
!pip install ucimlrepo
from ucimlrepo import fetch_ucirepo
from collections import Counter

bank = fetch_ucirepo(id=222).data.features
jobs = list(bank["job"])

job_counts = Counter(jobs)
print(job_counts.most_common(3))
print(job_counts["student"])
print(job_counts["astronaut"])

### defaultdict

In [ ]:
from collections import defaultdict

customers = [("Priya", "technician"), ("Arjun", "management"),
             ("Meera", "management"), ("Ravi", "technician"), ("Sana", "student")]

names_by_job = defaultdict(list)
for name, job in customers:
    names_by_job[job].append(name)

print(dict(names_by_job))

### namedtuple

In [ ]:
from collections import namedtuple

Customer = namedtuple("Customer", ["name", "age", "balance"])
priya = Customer("Priya", 33, 1506)

print(priya.name, priya.balance)
print(priya[1])
print(priya)

### itertools

In [ ]:
from itertools import combinations, product, accumulate, islice

products = ["term deposit", "credit card", "home loan"]
for pair in combinations(products, 2):
    print(pair)

for channel, time in product(["mobile", "landline"], ["morning", "evening"]):
    print(channel, time)

deposits = [500, 1200, 300]
print(list(accumulate(deposits)))
print(list(islice(range(100), 5)))

## 3. Scope in depth
### LEGB

In [ ]:
x = "global"

def outer():
    x = "enclosing"
    def inner():
        x = "local"
        print("inner sees:", x)
    inner()
    print("outer sees:", x)

outer()
print("module sees:", x)

### global and nonlocal

In [ ]:
# This version fails on purpose:
# calls_today = 0
# def make_call():
#     calls_today += 1
# make_call()   # UnboundLocalError: cannot access local variable 'calls_today' ...

calls_today = 0

def make_call():
    global calls_today
    calls_today += 1

make_call()
make_call()
print(calls_today)

In [ ]:
def make_counter():
    count = 0
    def add_call():
        nonlocal count
        count += 1
        return count
    return add_call

counter = make_counter()
print(counter(), counter(), counter())

## 4. Recursion
### A countdown

In [ ]:
def countdown(n):
    if n == 0:
        print("Lift off!")
        return
    print(n)
    countdown(n - 1)

countdown(3)

### Factorial

In [ ]:
def factorial(n):
    if n <= 1:
        return 1
    return n * factorial(n - 1)

print(factorial(5))

### Nested data

In [ ]:
def nested_sum(items):
    total = 0
    for item in items:
        if isinstance(item, list):
            total += nested_sum(item)
        else:
            total += item
    return total

balances = [2143, [29, 2], [1506, [1]]]
print(nested_sum(balances))

In [ ]:
import sys
print(sys.getrecursionlimit())

## 5. Functions as objects, closures and decorators
### Step 1: Functions are objects

In [ ]:
def shout(text):
    return text.upper()

speak = shout
print(speak("subscribed"))

def apply_to_all(func, items):
    return [func(item) for item in items]

print(apply_to_all(shout, ["yes", "no"]))
print(apply_to_all(len, ["management", "student"]))

### Step 2: Closures

In [ ]:
def make_interest_calculator(rate):
    def calculate(balance):
        return round(balance * (1 + rate), 2)
    return calculate

fd_7_percent = make_interest_calculator(0.07)
savings_3_percent = make_interest_calculator(0.03)

print(fd_7_percent(1000))
print(savings_3_percent(1000))

### Step 3: Decorators (your time will differ)

In [ ]:
import time
import functools

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        seconds = time.perf_counter() - start
        print(f"{func.__name__} took {seconds:.4f} seconds")
        return result
    return wrapper

@timer
def count_low_balances(balances, limit):
    return sum(1 for b in balances if b < limit)

print(count_low_balances(list(range(-500, 1_000_000)), 500))

In [ ]:
def log_calls(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__} with {args}")
        return func(*args, **kwargs)
    return wrapper

@log_calls
def add_interest(balance, rate):
    return round(balance * (1 + rate), 2)

print(add_interest(2143, 0.07))

### Built-in decorators: lru_cache

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=None)
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

print(fib(80))

## 6. Iterators and generators
### How a for loop really works

In [ ]:
balances = [2143, 29, 2]
it = iter(balances)
print(next(it))
print(next(it))
print(next(it))
try:
    next(it)
except StopIteration:
    print("No more items")

### Generators

In [ ]:
def first_customers(n):
    for i in range(1, n + 1):
        yield f"Customer {i}"

gen = first_customers(3)
print(gen)
print(next(gen))
for name in gen:
    print(name)

### Reading a big file lazily
First save the data as a CSV file (needs an internet connection).

In [ ]:
!pip install ucimlrepo
from ucimlrepo import fetch_ucirepo

bank = fetch_ucirepo(id=222).data.features
bank.to_csv("bank_customers.csv", index=False)

In [ ]:
def read_customers(path):
    with open(path) as file:
        next(file)
        for line in file:
            values = line.strip().split(",")
            yield int(values[0]), int(values[5])

for age, balance in read_customers("bank_customers.csv"):
    if balance < 0:
        print("First overdrawn customer: age", age, "balance", balance)
        break

### Generator expressions (exact sizes vary between Python versions)

In [ ]:
import sys

squares_list = [n * n for n in range(1_000_000)]
squares_gen = (n * n for n in range(1_000_000))
print(sys.getsizeof(squares_list), sys.getsizeof(squares_gen))
print(sum(squares_gen))
print(sum(squares_gen))

## 7. Advanced OOP
### Magic methods

In [ ]:
class Account:
    def __init__(self, owner, balance):
        self.owner = owner
        self.balance = balance

    def __repr__(self):
        return f"Account(owner={self.owner!r}, balance={self.balance})"

    def __eq__(self, other):
        return self.owner == other.owner and self.balance == other.balance

    def __lt__(self, other):
        return self.balance < other.balance

    def __add__(self, other):
        return Account(f"{self.owner} & {other.owner}", self.balance + other.balance)

a = Account("Priya", 1506)
b = Account("Arjun", 2143)
print(a)
print([a, b])
print(a == Account("Priya", 1506))
print(a < b)
print(sorted([b, a]))
print(a + b)

In [ ]:
class Portfolio:
    def __init__(self, accounts):
        self.accounts = accounts

    def __len__(self):
        return len(self.accounts)

    def __getitem__(self, position):
        return self.accounts[position]

p = Portfolio([a, b])
print(len(p))
print(p[0])
for acc in p:
    print(acc.owner)

### @property

In [ ]:
class BankCustomer:
    def __init__(self, name, balance):
        self.name = name
        self._balance = balance

    @property
    def balance(self):
        return self._balance

    @balance.setter
    def balance(self, value):
        if value < -1000:
            raise ValueError("Overdraft limit is -1,000 euros")
        self._balance = value

    @property
    def is_overdrawn(self):
        return self._balance < 0

priya = BankCustomer("Priya", 1506)
print(priya.balance)
priya.balance = -200
print(priya.balance, priya.is_overdrawn)
try:
    priya.balance = -5000
except ValueError as error:
    print("Error:", error)

### @classmethod and @staticmethod

In [ ]:
class BankCustomer:
    customers_created = 0

    def __init__(self, name, age, balance):
        self.name = name
        self.age = age
        self.balance = balance
        BankCustomer.customers_created += 1

    @classmethod
    def from_csv_row(cls, row):
        values = row.split(",")
        return cls(values[0], int(values[1]), int(values[2]))

    @staticmethod
    def is_valid_age(age):
        return 18 <= age <= 120

meera = BankCustomer.from_csv_row("Meera,47,102127")
print(meera.name, meera.balance)
print(BankCustomer.is_valid_age(214))
print(BankCustomer.customers_created)

### Dataclasses

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Customer:
    name: str
    age: int
    balance: float = 0.0
    loans: list[str] = field(default_factory=list)

priya = Customer("Priya", 33, 1506)
sana = Customer("Sana", 21)
print(priya)
print(sana)
print(priya == Customer("Priya", 33, 1506))
sana.loans.append("student")
print(sana.loans, priya.loans)

### Abstract classes

In [ ]:
from abc import ABC, abstractmethod

class Product(ABC):
    @abstractmethod
    def monthly_cost(self):
        pass

class TermDeposit(Product):
    def __init__(self, amount):
        self.amount = amount

    def monthly_cost(self):
        return 0

class CreditCard(Product):
    def monthly_cost(self):
        return 2.5

for item in [TermDeposit(5000), CreditCard()]:
    print(type(item).__name__, item.monthly_cost())

try:
    Product()
except TypeError as error:
    print("TypeError:", error)

### Multiple inheritance and the MRO

In [ ]:
class Saver:
    def describe(self):
        return "Saver: likes deposits"

    def interest_bonus(self):
        return 0.5

class Borrower:
    def describe(self):
        return "Borrower: has loans"

    def loan_count(self):
        return 2

class MixedCustomer(Saver, Borrower):
    pass

m = MixedCustomer()
print(m.interest_bonus(), m.loan_count())
print(m.describe())
print([c.__name__ for c in MixedCustomer.__mro__])

## 8. Context managers
### Method 1: a class (your time will differ)

In [ ]:
import time

class Timer:
    def __enter__(self):
        self.start = time.perf_counter()
        print("Timer started")
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        self.seconds = time.perf_counter() - self.start
        print(f"Timer stopped after {self.seconds:.3f} seconds")
        return False

with Timer() as t:
    total = sum(range(5_000_000))
print(total)

### Method 2: @contextmanager

In [ ]:
from contextlib import contextmanager

@contextmanager
def temporary_balance(customer, new_balance):
    original = customer["balance"]
    customer["balance"] = new_balance
    try:
        yield customer
    finally:
        customer["balance"] = original

priya = {"name": "Priya", "balance": 1506}
with temporary_balance(priya, 50000) as c:
    print("Inside:", c["balance"])
print("After:", priya["balance"])

In [ ]:
try:
    with temporary_balance(priya, 99999):
        raise ValueError("Simulated crash")
except ValueError:
    pass
print("After a crash:", priya["balance"])

## 9. Regular expressions and pathlib
### Finding patterns

In [ ]:
import re

note = "Customer Priya called on 05/05/2026, mobile 98765-43210, balance EUR 1,506.50"

print(re.search(r"\d+", note).group())
print(re.findall(r"\d+", note))
print(re.findall(r"\d{2}/\d{2}/\d{4}", note))
phone = re.search(r"\d{5}-\d{5}", note)
print(phone.group())
print(re.sub(r"\d{5}-\d{5}", "XXXXX-XXXXX", note))

### Capturing part of a match

In [ ]:
amount = re.search(r"EUR ([\d,]+\.\d{2})", note)
print(amount.group(0))
print(amount.group(1))
print(float(amount.group(1).replace(",", "")))

### Cleaning messy categories

In [ ]:
messy = ["  Blue-Collar!! ", "ADMIN.", "self employed", "Self-Employed  "]
clean = [re.sub(r"[^a-z]+", "-", job.lower()).strip("-") for job in messy]
print(clean)

### Checking a format

In [ ]:
emails = ["priya@example.com", "arjun.at.example.com", "meera@bank.pt"]
pattern = re.compile(r"^[\w.]+@[\w]+\.[a-z]{2,}$")
for e in emails:
    print(e, bool(pattern.match(e)))

### pathlib

In [ ]:
from pathlib import Path

reports = Path("reports")
reports.mkdir(exist_ok=True)

summary_file = reports / "summary.txt"
summary_file.write_text("Customers: 45211\nSubscribed: 5289\n")

print(summary_file)
print(summary_file.exists())
print(summary_file.name, summary_file.suffix, summary_file.stem)
print(summary_file.read_text())
for path in sorted(Path(".").glob("*.csv")):
    print("CSV found:", path)

## 10. Interview traps
### is versus ==

In [ ]:
a = [2143, 29]
b = [2143, 29]
c = a
print(a == b)
print(a is b)
print(a is c)

result = None
print(result is None)

### The mutable default argument

In [ ]:
def add_call(note, history=[]):
    history.append(note)
    return history

print(add_call("Called Priya"))
print(add_call("Called Arjun"))

In [ ]:
def add_call(note, history=None):
    if history is None:
        history = []
    history.append(note)
    return history

print(add_call("Called Priya"))
print(add_call("Called Arjun"))

### Floating-point precision

In [ ]:
print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)

import math
print(math.isclose(0.1 + 0.2, 0.3))
print(round(0.1 + 0.2, 2) == 0.3)

from decimal import Decimal
print(Decimal("0.10") + Decimal("0.20"))

## 11. Testing
### assert

In [ ]:
def add_interest(balance, rate):
    return round(balance * (1 + rate), 2)

assert add_interest(1000, 0.07) == 1070.0
assert add_interest(0, 0.07) == 0
print("All checks passed")

# This one fails on purpose:
# assert add_interest(1000, 0.07) == 1700.0, "Interest calculation is wrong"

### pytest: the code to test

In [ ]:
%%writefile bank_utils.py
"""Helper functions for bank customer data."""


def add_interest(balance, rate):
    """Return the balance after one year of simple interest."""
    if rate < 0:
        raise ValueError("Rate cannot be negative")
    return round(balance * (1 + rate), 2)


def balance_label(balance):
    """Return 'Overdrawn', 'Low' or 'Healthy' for a balance."""
    if balance < 0:
        return "Overdrawn"
    if balance < 1000:
        return "Low"
    return "Healthy"

### pytest: the tests

In [ ]:
%%writefile test_bank_utils.py
import pytest
from bank_utils import add_interest, balance_label


def test_add_interest_normal():
    assert add_interest(1000, 0.07) == 1070.0


def test_add_interest_zero_balance():
    assert add_interest(0, 0.07) == 0


def test_add_interest_negative_rate_raises():
    with pytest.raises(ValueError):
        add_interest(1000, -0.05)


@pytest.mark.parametrize("balance, expected", [
    (-8019, "Overdrawn"),
    (448, "Low"),
    (999, "Low"),
    (1000, "Healthy"),
    (102127, "Healthy"),
])
def test_balance_label(balance, expected):
    assert balance_label(balance) == expected

In [ ]:
!python -m pytest -q test_bank_utils.py

**Try the failing test from the book:** in the `bank_utils.py` cell, change `balance < 1000` to `balance <= 1000`, run that cell again, then re-run pytest. Change it back afterwards.

## 12. Mini project: a text-cleaning toolkit
### Step 1: The toolkit module

In [ ]:
%%writefile text_cleaning.py
"""A small toolkit for cleaning messy customer text data."""
import re
from collections import Counter
from dataclasses import dataclass


def clean_category(text: str) -> str:
    """Lowercase text and join words with single hyphens.

    '  Blue Collar!! ' becomes 'blue-collar'.
    """
    cleaned = re.sub(r"[^a-z]+", "-", text.lower()).strip("-")
    if not cleaned:
        raise ValueError(f"No letters found in {text!r}")
    return cleaned


def parse_amount(text: str) -> float:
    """Extract a money amount such as 'EUR 1,506.50' as a float."""
    match = re.search(r"-?[\d,]+(\.\d+)?", text)
    if match is None:
        raise ValueError(f"No amount found in {text!r}")
    return float(match.group().replace(",", ""))


def mask_phone(text: str) -> str:
    """Replace 10-digit phone numbers (with or without a dash) by X's."""
    return re.sub(r"\b\d{5}-?\d{5}\b", "XXXXX-XXXXX", text)


@dataclass
class CleaningReport:
    rows: int
    variants_before: int
    categories_after: int


def clean_column(values: list[str]) -> tuple[list[str], CleaningReport]:
    """Clean every value in a column and report what changed."""
    cleaned = [clean_category(v) for v in values]
    report = CleaningReport(
        rows=len(values),
        variants_before=len(Counter(values)),
        categories_after=len(Counter(cleaned)),
    )
    return cleaned, report

### Step 2: The tests

In [ ]:
%%writefile test_text_cleaning.py
import pytest
from text_cleaning import clean_category, parse_amount, mask_phone, clean_column


@pytest.mark.parametrize("raw, expected", [
    ("blue-collar", "blue-collar"),
    ("  Blue Collar!! ", "blue-collar"),
    ("ADMIN.", "admin"),
    ("Self_Employed", "self-employed"),
])
def test_clean_category(raw, expected):
    assert clean_category(raw) == expected


def test_clean_category_rejects_text_without_letters():
    with pytest.raises(ValueError):
        clean_category("123 !!")


@pytest.mark.parametrize("raw, expected", [
    ("EUR 1,506.50", 1506.5),
    ("balance: 2143", 2143.0),
    ("-8019 overdrawn", -8019.0),
])
def test_parse_amount(raw, expected):
    assert parse_amount(raw) == expected


def test_parse_amount_rejects_text_without_numbers():
    with pytest.raises(ValueError):
        parse_amount("no money here")


def test_mask_phone():
    assert mask_phone("Call 98765-43210 or 9876543210") == "Call XXXXX-XXXXX or XXXXX-XXXXX"


def test_clean_column_report():
    cleaned, report = clean_column(["Admin.", "admin", " ADMIN "])
    assert cleaned == ["admin", "admin", "admin"]
    assert report.rows == 3
    assert report.variants_before == 3
    assert report.categories_after == 1

In [ ]:
!python -m pytest -q test_text_cleaning.py

### Step 3: Clean the real data (needs an internet connection)

In [ ]:
!pip install ucimlrepo
from ucimlrepo import fetch_ucirepo
from text_cleaning import clean_column

bank = fetch_ucirepo(id=222).data.features
jobs = list(bank["job"].fillna("unknown"))

def make_messy(job, i):
    if i % 3 == 0:
        return job.upper()
    if i % 5 == 0:
        return f"  {job}!! "
    if i % 7 == 0:
        return job.replace("-", " ").title()
    return job

messy_jobs = [make_messy(job, i) for i, job in enumerate(jobs)]
print(messy_jobs[:6])

cleaned, report = clean_column(messy_jobs)
print(cleaned[:6])
print(report)
print(sorted(set(cleaned)))

## Try it yourself
Use the empty cells below for the exercises at the end of Chapter 5.

1. Sets: compare the jobs of customers who subscribed with those who did not.
2. Counter: the three most common education levels and the least common marital status.
3. Recursion: `count_items([1, [2, 3], [4, [5, 6]]])` should return 6.
4. Decorator: `@count_calls` that stores the call count in `wrapper.calls`.
5. Generator: `batches(items, size)`; test on `range(10)` with size 3.
6. Rewrite the Chapter 4 `BankCustomer` as a dataclass with an `is_overdrawn` property.
7. A context manager that changes directory and always changes back.
8. Extract every `dd/mm/yyyy` date from text and return `date` objects.
9. Add three more tests to `test_text_cleaning.py`, including an edge case.

---
**Dataset credit:** S. Moro, P. Rita and P. Cortez, *Bank Marketing* dataset, UCI Machine Learning Repository (2014), https://doi.org/10.24432/C5K306. Licensed under CC BY 4.0.